# DecoyVerse ML Model — Accuracy Evaluation Report

| | |
|---|---|
| **Model** | Random Forest Classifier (100 trees, max_depth=15) |
| **Anomaly Detector** | Isolation Forest (contamination=0.1) |
| **Dataset** | 15,000 synthetic security log samples (training_data_v2.csv) |
| **Task** | Multi-class attack type classification (5 classes) |
| **Evaluation** | 80/20 stratified train/test split (random_state=42) |

---

## Input Features (6 features extracted per event)

| Feature | Type | Range | Meaning |
|---|---|---|---|
| `failed_logins` | int | 0–199 | Number of failed authentication attempts in the session |
| `request_rate` | int | 1–1999 | HTTP/network requests per second |
| `commands_count` | int | 0–20 | Shell/OS commands executed in the session |
| `sql_payload` | binary | 0 or 1 | SQL injection pattern detected (SELECT/UNION/DROP/1=1) |
| `honeytoken_access` | binary | 0 or 1 | A fake credential file (honeytoken) was opened |
| `session_time` | int | 5–3598 | Total session duration in seconds |

## Output (model prediction)

| Output | Values | Description |
|---|---|---|
| `attack_type` | Normal, BruteForce, Injection, DataExfil, Recon | Classified attack category |
| `risk_score` | 1–10 | Computed severity score (higher = more dangerous) |
| `confidence` | 0–1 | Model's certainty in the prediction |
| `is_anomaly` | True/False | Isolation Forest anomaly flag |

In [ ]:
# Cell 1 — Install seaborn if not present
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seaborn', '-q'])

In [ ]:
# Cell 2 — Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
    roc_curve, auc, accuracy_score
)

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#3b82f6', '#ef4444', '#f59e0b', '#10b981', '#8b5cf6']
print('All libraries loaded successfully.')

In [ ]:
# Cell 3 — Load dataset
df = pd.read_csv('training_data_v2.csv')

FEATURES = ['failed_logins', 'request_rate', 'commands_count',
            'sql_payload', 'honeytoken_access', 'session_time']

X = df[FEATURES].values
y = df['label'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)
CLASS_NAMES = le.classes_

print(f'Dataset shape   : {df.shape}')
print(f'Features        : {FEATURES}')
print(f'Classes         : {CLASS_NAMES}')
print(f'Class counts    :')
print(pd.Series(y).value_counts().to_string())

In [ ]:
# Cell 4 — Train/Test Split and Model Training
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# Random Forest Classifier (same hyperparameters as production)
clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
clf.fit(X_train, y_train)

# Isolation Forest for anomaly detection
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

iso = IsolationForest(contamination=0.1, n_estimators=100, random_state=42)
iso.fit(X_train_scaled)

y_pred   = clf.predict(X_test)
y_proba  = clf.predict_proba(X_test)
iso_pred = iso.predict(X_test_scaled)  # -1 = anomaly, 1 = normal

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc  = accuracy_score(y_test, y_pred)

print(f'Train size : {len(X_train)} samples')
print(f'Test size  : {len(X_test)} samples')
print(f'Train Accuracy : {train_acc:.4f}  ({train_acc*100:.2f}%)')
print(f'Test  Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

---
## Graph 1 — Confusion Matrix

A confusion matrix shows how many test samples were **correctly classified** (diagonal) vs **misclassified** (off-diagonal). 
Each row = actual class, each column = predicted class. Dark blue diagonal = high accuracy.

In [ ]:
# Graph 1 — Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_title('Graph 1 — Confusion Matrix: Attack Type Classification',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('graph1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph1_confusion_matrix.png')

---
## Graph 2 — Per-Class Precision, Recall & F1 Score

- **Precision**: Of all events the model labelled as attack X, how many actually were?
- **Recall**: Of all real attack X events, how many did the model catch?
- **F1 Score**: Harmonic mean of Precision and Recall (best overall metric per class)

In [ ]:
# Graph 2 — Per-Class Precision / Recall / F1
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred)

x = np.arange(len(CLASS_NAMES))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
bars_p = ax.bar(x - w, precision, w, label='Precision', color='#3b82f6', edgecolor='white')
bars_r = ax.bar(x,     recall,    w, label='Recall',    color='#ef4444', edgecolor='white')
bars_f = ax.bar(x + w, f1,        w, label='F1 Score',  color='#10b981', edgecolor='white')

# Value labels
for bar in list(bars_p) + list(bars_r) + list(bars_f):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
            f'{h:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score (0–1)', fontsize=12)
ax.set_title('Graph 2 — Per-Class Precision, Recall & F1 Score',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('graph2_per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph2_per_class_metrics.png')

---
## Graph 3 — Feature Importance

Random Forest computes **how much each input feature contributed** to splitting decisions across all 100 trees.
A higher score = more important for predicting attack types.

In [ ]:
# Graph 3 — Feature Importance
importances = clf.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]
sorted_features    = [FEATURES[i] for i in sorted_idx]
sorted_importances = importances[sorted_idx]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(FEATURES)), sorted_importances,
              color=COLORS[:len(FEATURES)], edgecolor='white')

for bar, val in zip(bars, sorted_importances):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels(sorted_features, rotation=25, ha='right', fontsize=11)
ax.set_ylabel('Importance Score', fontsize=12)
ax.set_title('Graph 3 — Feature Importance (Random Forest)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graph3_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph3_feature_importance.png')
print()
for f, imp in zip(sorted_features, sorted_importances):
    print(f'  {f:<22} {imp:.4f}')

---
## Graph 4 — ROC Curves (One-vs-Rest, all classes)

The **ROC curve** plots True Positive Rate vs False Positive Rate at every threshold.
**AUC (Area Under Curve)** = 1.0 is perfect; > 0.95 is excellent. One curve per attack class.

In [ ]:
# Graph 4 — ROC Curves
y_test_bin = label_binarize(y_test, classes=range(len(CLASS_NAMES)))

fig, ax = plt.subplots(figsize=(8, 6))

for i, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f'{name}  (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Graph 4 — ROC Curves (One-vs-Rest, all attack classes)',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('graph4_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph4_roc_curves.png')

---
## Graph 5 — Training Data Class Distribution

Shows how many samples of each attack type exist in the dataset.
The **class imbalance** (60% Normal, 5% Injection) reflects real-world distributions — most traffic is normal.

In [ ]:
# Graph 5 — Class Distribution
counts = pd.Series(y).value_counts()
# Sort by CLASS_NAMES order
counts = counts.reindex(CLASS_NAMES).fillna(0).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = ax1.bar(counts.index, counts.values, color=COLORS, edgecolor='white', width=0.6)
for bar, val in zip(bars, counts.values):
    pct = val / len(y) * 100
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f'{val}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('Sample Count', fontsize=12)
ax1.set_title('Class Distribution (Bar)', fontsize=13, fontweight='bold')
ax1.set_xticklabels(counts.index, fontsize=11)

# Pie chart
wedges, texts, autotexts = ax2.pie(
    counts.values, labels=counts.index, colors=COLORS,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(10)
ax2.set_title('Class Distribution (Pie)', fontsize=13, fontweight='bold')

fig.suptitle('Graph 5 — Training Data Class Distribution (1,000 samples total)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('graph5_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph5_class_distribution.png')

---
## Graph 6 — Train vs Test Accuracy

Compares accuracy on training data vs held-out test data.
If train accuracy >> test accuracy, the model is **overfitting**. Similar values = healthy generalisation.

In [ ]:
# Graph 6 — Train vs Test Accuracy
fig, ax = plt.subplots(figsize=(6, 5))

bar_colors = ['#3b82f6', '#10b981']
bars = ax.bar(['Train Accuracy', 'Test Accuracy'],
              [train_acc, test_acc],
              color=bar_colors, width=0.45, edgecolor='white')

for bar, val in zip(bars, [train_acc, test_acc]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val * 100:.2f}%', ha='center', va='bottom',
            fontsize=14, fontweight='bold')

ax.set_ylim(0, 1.12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Graph 6 — Train vs Test Accuracy\n(overfitting check)',
             fontsize=13, fontweight='bold')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('graph6_train_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph6_train_vs_test.png')
gap = abs(train_acc - test_acc)
print(f'Accuracy gap: {gap*100:.2f}%  ({"minimal — no overfitting" if gap < 0.05 else "some overfitting detected"})')

---
## Graph 7 — Risk Score Distribution by Attack Type

Shows the range of **risk scores (1–10)** assigned to each attack type.
The scoring formula multiplies ML confidence by type-specific weights (DataExfil = 2.0×, Injection = 1.5×, etc.)
so higher-severity attacks always produce higher risk scores.

In [ ]:
# Graph 7 — Risk Score Distribution by Attack Type
MULTIPLIERS = {
    'DataExfil': 2.0,
    'Injection': 1.5,
    'BruteForce': 1.2,
    'Recon': 0.8,
    'Normal': 0.3
}

# Pre-compute iso scores once (using already-scaled X_test_scaled from Cell 4)
iso_scores = iso.score_samples(X_test_scaled)

risk_by_class = {name: [] for name in CLASS_NAMES}

for i, (pred_idx, proba_row) in enumerate(zip(y_pred, y_proba)):
    name = CLASS_NAMES[pred_idx]
    conf = proba_row[pred_idx]
    base = conf * 6 + abs(iso_scores[i]) * 4
    score = min(base * MULTIPLIERS[name], 10.0)
    score = max(score, 1.0)
    risk_by_class[name].append(score)

# Box plot
data_to_plot = [risk_by_class[n] for n in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    data_to_plot, labels=CLASS_NAMES,
    patch_artist=True, notch=False,
    medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Risk Score (1–10)', fontsize=12)
ax.set_xlabel('Attack Type', fontsize=12)
ax.set_ylim(0, 11)
ax.axhline(y=7, color='red', linestyle='--', alpha=0.6, label='Alert threshold (≥7)')
ax.legend(fontsize=10)
ax.set_title('Graph 7 — Risk Score Distribution by Attack Type',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graph7_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graph7_risk_distribution.png')

---
## Summary

In [ ]:
# Final summary table
from sklearn.metrics import precision_recall_fscore_support
p, r, f, sup = precision_recall_fscore_support(y_test, y_pred)
auc_scores = []
y_test_bin2 = label_binarize(y_test, classes=range(len(CLASS_NAMES)))
for i in range(len(CLASS_NAMES)):
    fpr, tpr, _ = roc_curve(y_test_bin2[:, i], y_proba[:, i])
    auc_scores.append(auc(fpr, tpr))

print('=' * 65)
print('           DECOYVERSE ML MODEL — FINAL ACCURACY SUMMARY')
print('=' * 65)
print(f'  Train Accuracy  : {train_acc*100:.2f}%')
print(f'  Test  Accuracy  : {test_acc*100:.2f}%')
print(f'  Overfitting Gap : {abs(train_acc-test_acc)*100:.2f}%')
print()
print(f'  {"Class":<14} {"Precision":>10} {"Recall":>8} {"F1":>8} {"AUC":>8} {"Support":>9}')
print('  ' + '-' * 59)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<14} {p[i]:>10.3f} {r[i]:>8.3f} {f[i]:>8.3f} {auc_scores[i]:>8.3f} {sup[i]:>9}')
print('=' * 65)
print()
print('Most important feature :', FEATURES[np.argmax(clf.feature_importances_)])
print('Least important feature:', FEATURES[np.argmin(clf.feature_importances_)])
print()
anomaly_count = (iso_pred == -1).sum()
print(f'Isolation Forest detected {anomaly_count}/{len(X_test)} test samples as anomalous ({anomaly_count/len(X_test)*100:.1f}%)')
print()
print('Graph files saved:')
for i in range(1, 8):
    names = ['confusion_matrix','per_class_metrics','feature_importance',
             'roc_curves','class_distribution','train_vs_test','risk_distribution']
    print(f'  graph{i}_{names[i-1]}.png')

---

## How to Use These Graphs in Your Report

| Graph | Where to use it in the report |
|---|---|
| Graph 1 (Confusion Matrix) | Section: Model Evaluation / Results |
| Graph 2 (Precision/Recall/F1) | Section: Per-Class Performance |
| Graph 3 (Feature Importance) | Section: Feature Engineering / What the model learned |
| Graph 4 (ROC Curves) | Section: Model Evaluation (advanced metric) |
| Graph 5 (Class Distribution) | Section: Dataset Description |
| Graph 6 (Train vs Test) | Section: Model Training / Overfitting Analysis |
| Graph 7 (Risk Score Distribution) | Section: Risk Scoring System |

### How to export this notebook as PDF or HTML:
```bash
# HTML (easiest, opens in any browser)
jupyter nbconvert --to html model_accuracy_report.ipynb

# PDF (requires latex installed)
jupyter nbconvert --to pdf model_accuracy_report.ipynb

# Or: In Jupyter UI → File → Download as → HTML / PDF
```

### To paste graphs into Word / Google Docs:
All 7 PNG files are saved in the same folder as this notebook. Insert → Image → choose any `graph*.png`.